# Module 02: Model Parallelism

Read `README.md` before starting.

**Strategy**: Different layers on different GPUs. Solves memory, not speed.

## Part 1: Demonstrate the OOM Problem

In [ ]:
import torch
import torch.nn as nn

# Build a model large enough to cause memory pressure.
# The Large config requires ~14GB for training (params + grads + optimizer).
# With activations, it will OOM on a 16GB T4.

def make_large_model(d_model=1024, num_layers=24, vocab_size=50000):
    """~350M parameter transformer. Tight fit on one T4."""
    encoder_layer = nn.TransformerEncoderLayer(
        d_model, nhead=16, dim_feedforward=d_model*4, 
        dropout=0.1, batch_first=True
    )
    return nn.Sequential(
        nn.Embedding(vocab_size, d_model),
        nn.TransformerEncoder(encoder_layer, num_layers),
        nn.Linear(d_model, vocab_size)
    )

# Count parameters
model = make_large_model()
total_params = sum(p.numel() for p in model.parameters())
param_mem_fp16 = total_params * 2 / 1e9
full_training_mem = total_params * 16 / 1e9

print(f"Total parameters:    {total_params:,}")
print(f"Params (fp16):       {param_mem_fp16:.1f} GB")
print(f"Full training mem:   {full_training_mem:.1f} GB (params+grads+optimizer)")
print(f"T4 VRAM:             16.0 GB")
print()

if full_training_mem > 16:
    print("→ This model CANNOT train on a single T4.")
else:
    print("→ Params fit, but activations + overhead will OOM.")

del model
torch.cuda.empty_cache()

In [ ]:
# Try to load on single GPU — watch it fail (or come very close)
print("Attempting to load large model on GPU 0 only...")
print(f"GPU 0 free VRAM before: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.1f} GB")

try:
    model_single = make_large_model().to('cuda:0')
    mem_used = torch.cuda.memory_allocated(0) / 1e9
    print(f"Loaded! GPU 0 used: {mem_used:.1f} GB")
    print("Now try a forward pass with batch_size=8...")
    
    x = torch.randint(0, 50000, (8, 256)).to('cuda:0')
    with torch.no_grad():
        out = model_single(x)
    print(f"Forward pass OK. Output shape: {out.shape}")
    print(f"GPU 0 after forward: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")
    
except torch.cuda.OutOfMemoryError as e:
    print(f"OOM! {e}")
    print("This is exactly the problem model parallelism solves.")
finally:
    try:
        del model_single, x, out
    except:
        pass
    torch.cuda.empty_cache()

## Part 2: Model Parallelism — Layer Split

In [ ]:
import torch
import torch.nn as nn

class ModelParallelTransformer(nn.Module):
    """
    A transformer split across 2 GPUs.

    GPU 0: Embedding + first half of transformer layers
    GPU 1: Second half of transformer layers + output head

    The only non-obvious thing: inputs must start on the same device
    as the first layer. Outputs live on the last layer's device.
    """

    def __init__(
        self,
        vocab_size: int = 50000,
        d_model: int = 1024,
        nhead: int = 16,
        num_layers: int = 24,
    ):
        super().__init__()
        split = num_layers // 2  # 12 layers per GPU

        # ── GPU 0 ──────────────────────────────────────────────────────────────
        self.embedding = nn.Embedding(vocab_size, d_model).to('cuda:0')

        # Build layers individually so we can assign them to GPUs
        self.layers_gpu0 = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model, nhead, dim_feedforward=d_model*4,
                dropout=0.1, batch_first=True
            ).to('cuda:0')
            for _ in range(split)
        ])

        # ── GPU 1 ──────────────────────────────────────────────────────────────
        self.layers_gpu1 = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model, nhead, dim_feedforward=d_model*4,
                dropout=0.1, batch_first=True
            ).to('cuda:1')
            for _ in range(split)
        ])
        self.output_head = nn.Linear(d_model, vocab_size).to('cuda:1')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input: token ids, shape [batch, seq_len]
        # Assumes input is on CPU or will be moved to cuda:0

        x = x.to('cuda:0')
        x = self.embedding(x)            # [batch, seq_len, d_model] on GPU 0

        for layer in self.layers_gpu0:
            x = layer(x)                  # Still on GPU 0

        # ← Cross-GPU transfer happens here.
        # PyTorch autograd tracks this — gradients will flow back automatically.
        x = x.to('cuda:1')

        for layer in self.layers_gpu1:
            x = layer(x)                  # Now on GPU 1

        # Language model head: predict next token for each position
        logits = self.output_head(x)     # [batch, seq_len, vocab_size] on GPU 1
        return logits


# Build the model
print("Building model across 2 GPUs...")
torch.cuda.empty_cache()

model = ModelParallelTransformer(
    vocab_size=50000,
    d_model=1024,
    nhead=16,
    num_layers=24
)

# Check memory distribution
print()
for i in range(2):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {allocated:.2f} GB / {total:.0f} GB ({100*allocated/total:.0f}% used)")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal params: {total_params:,}")
print("Model loaded successfully across both GPUs.")

In [ ]:
import torch.optim as optim
import time

# Training loop
# IMPORTANT: Loss must be computed on the same device as outputs (cuda:1)
# Labels must also be on cuda:1

optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

BATCH_SIZE = 4
SEQ_LEN = 128
VOCAB_SIZE = 50000

throughputs = []

print("Training (model parallel)...")
for step in range(10):
    t0 = time.perf_counter()

    # Input lives on CPU; model.forward() moves it to cuda:0 internally
    x = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    # Labels: shifted input (next-token prediction)
    y = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE * SEQ_LEN,)).to('cuda:1')

    optimizer.zero_grad()
    out = model(x)                        # Flows: CPU → GPU0 → GPU1
    out_flat = out.view(-1, VOCAB_SIZE)   # Still on GPU 1
    loss = criterion(out_flat, y)         # Both on GPU 1
    loss.backward()                       # Flows: GPU1 → GPU0
    optimizer.step()

    elapsed = time.perf_counter() - t0
    tp = BATCH_SIZE / elapsed
    throughputs.append(tp)

    mem_0 = torch.cuda.memory_allocated(0) / 1e9
    mem_1 = torch.cuda.memory_allocated(1) / 1e9
    print(f"[Step {step}] loss={loss.item():.4f} | {tp:.1f} samples/s | GPU0={mem_0:.1f}GB GPU1={mem_1:.1f}GB")

avg_tp = sum(throughputs) / len(throughputs)
print(f"\nAvg throughput: {avg_tp:.1f} samples/s")
print("Compare this to your Module 00 single-GPU baseline.")
print("Model parallelism does NOT improve speed — it just enables bigger models.")

## Part 3: See the Bubble Problem

Let's visualize GPU utilization to see the idle time.

In [ ]:
import subprocess
import threading
import time

# We'll sample GPU utilization in a background thread while training runs
gpu_utils = {0: [], 1: []}
stop_sampling = threading.Event()

def sample_gpu_util():
    while not stop_sampling.is_set():
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            utils = result.stdout.strip().split('\n')
            for i, u in enumerate(utils[:2]):
                try:
                    gpu_utils[i].append(int(u.strip()))
                except ValueError:
                    pass
        time.sleep(0.1)

# Start sampling
sampler = threading.Thread(target=sample_gpu_util, daemon=True)
sampler.start()

# Run training for a few steps
for step in range(5):
    x = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    y = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE * SEQ_LEN,)).to('cuda:1')
    optimizer.zero_grad()
    out = model(x)
    loss = criterion(out.view(-1, VOCAB_SIZE), y)
    loss.backward()
    optimizer.step()

stop_sampling.set()
time.sleep(0.2)

# Report
if gpu_utils[0] and gpu_utils[1]:
    avg_0 = sum(gpu_utils[0]) / len(gpu_utils[0])
    avg_1 = sum(gpu_utils[1]) / len(gpu_utils[1])
    print(f"Average GPU utilization during model-parallel training:")
    print(f"  GPU 0: {avg_0:.0f}%")
    print(f"  GPU 1: {avg_1:.0f}%")
    print()
    print("Both GPUs should be well below 100%.")
    print("This is the 'bubble problem' — GPUs waiting for each other.")
    print("Module 03 (Pipeline Parallelism) will fix this.")
else:
    print("Could not sample GPU utilization. Check nvidia-smi is available.")

## Part 4: Balancing the Split

The split point matters. If GPU 0 has 90% of the layers, GPU 1 sits idle waiting.
The optimal split minimizes max(GPU 0 time, GPU 1 time).

In [ ]:
import time

# Measure how long each stage takes for different split points
D_MODEL = 512
VOCAB = 10000
NUM_LAYERS = 8
BATCH = 4
SEQ = 128

def make_transformer_encoder_layers(n, d_model, nhead, device):
    layers = nn.ModuleList([
        nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model*4, batch_first=True).to(device)
        for _ in range(n)
    ])
    return layers

print(f"{'Split':>10} {'GPU0 time':>12} {'GPU1 time':>12} {'Total time':>12} {'Utilization':>14}")
print("-" * 65)

for split in [2, 4, 6]:
    torch.cuda.empty_cache()
    
    emb = nn.Embedding(VOCAB, D_MODEL).to('cuda:0')
    layers_0 = make_transformer_encoder_layers(split, D_MODEL, 8, 'cuda:0')
    layers_1 = make_transformer_encoder_layers(NUM_LAYERS - split, D_MODEL, 8, 'cuda:1')
    head = nn.Linear(D_MODEL, VOCAB).to('cuda:1')

    x = torch.randint(0, VOCAB, (BATCH, SEQ)).to('cuda:0')
    
    # Warm up
    with torch.no_grad():
        h = emb(x)
        for l in layers_0: h = l(h)
        h = h.to('cuda:1')
        for l in layers_1: h = l(h)
        _ = head(h)
    
    # Measure GPU 0 stage time
    torch.cuda.synchronize(0)
    t0 = time.perf_counter()
    with torch.no_grad():
        h = emb(x)
        for l in layers_0: h = l(h)
    torch.cuda.synchronize(0)
    t0_time = (time.perf_counter() - t0) * 1000
    
    # Measure GPU 1 stage time
    h = h.to('cuda:1')
    torch.cuda.synchronize(1)
    t1 = time.perf_counter()
    with torch.no_grad():
        for l in layers_1: h = l(h)
        _ = head(h)
    torch.cuda.synchronize(1)
    t1_time = (time.perf_counter() - t1) * 1000
    
    total = t0_time + t1_time
    utilization = min(t0_time, t1_time) / max(t0_time, t1_time) * 100
    print(f"{split:>4}/{NUM_LAYERS-split:<5} {t0_time:>10.1f}ms {t1_time:>10.1f}ms {total:>10.1f}ms {utilization:>13.0f}%")
    
    del emb, layers_0, layers_1, head
    torch.cuda.empty_cache()

print()
print("The split with highest utilization % is the most balanced.")
print("In practice, you need to profile to find the optimal split point.")

## Checkpoint Questions

1. In `ModelParallelTransformer.forward()`, what line moves tensors from GPU 0 to GPU 1?
2. Why does model parallelism NOT improve throughput compared to single-GPU training?
3. If you have a 4-layer model split 3/1 (GPU0 gets 3 layers, GPU1 gets 1), which GPU is the bottleneck?
4. What does PyTorch autograd do when a tensor moves across devices? Does backward still work?

**Answers:**
1. `x = x.to('cuda:1')` — this is the only line that crosses the GPU boundary.
2. Because only one GPU is active at a time. The other is always idle waiting.
3. GPU 0 is the bottleneck — it runs 3x more compute. GPU 1 is idle most of the time.
4. PyTorch tracks device transitions in the autograd graph. Backward automatically moves gradients back across the device boundary.

---

## Next: Open `../03_pipeline_parallelism/notebook.ipynb`